In [12]:
import aiohttp
import bs4
import orjson
import datetime
import dataclasses
import time
import os
import traceback
import asyncio
import typing
import aiofiles

@dataclasses.dataclass
class Price24K:
    buy_bullion: int
    sell_bullion: int
    buy_ring: int
    sell_ring: int
    taken_on: datetime.datetime
    vendor: str = ""

In [13]:
async def sjc_today(session: aiohttp.ClientSession) -> Price24K:
    buy_bullion: int = 0
    sell_bullion: int = 0
    buy_ring: int = 0
    sell_ring: int = 0
    # TODO - temporary countermeasure as SJC uses Cloudflare
    # with requests.Session() as s:
    #     data = s.get("https://sjc.com.vn/GoldPrice/Services/PriceService.ashx")
    #     for i in data["data"]:
    #         if i["TypeName"] == "Vàng SJC 1L, 10L, 1KG" and i["BranchName"] == "Hồ Chí Minh":
    #             buy_bullion = int(i["BuyValue"])
    #             sell_bullion = int(i["SellValue"])
    #         if i["TypeName"] == "Vàng nhẫn SJC 99,99% 1 chỉ, 2 chỉ, 5 chỉ" and i["BranchName"] == "Hồ Chí Minh":
    #             buy_ring = int(i["BuyValue"])
    #             sell_ring = int(i["SellValue"])
    #     return Price24K(
    #         buy_bullion, sell_bullion, buy_ring, sell_ring, taken_on=datetime.datetime.now(), vendor="SJC"
    #     )
    data = await (await session.get("https://edge-api.pnj.io/ecom-frontend/v1/get-gold-price?zone=00")).json()
    for i in data["data"]:
        if i["masp"] == "SJC":
            sell_ring = sell_bullion = i["giaban"]*10000
            buy_ring = buy_bullion = i["giamua"]*10000
    return Price24K(
        buy_bullion, sell_bullion, buy_ring, sell_ring, taken_on=datetime.datetime.now(), vendor="SJC"
    )

In [14]:
async def doji_today(session: aiohttp.ClientSession) -> Price24K:
    req = await session.get("https://giavang.doji.vn/")
    soup = bs4.BeautifulSoup(await req.text())
    buy_bullion, sell_bullion, buy_ring, sell_ring = [int(list(i.children)[0].text.replace(",", "")) * 10_000 for i in soup.select(".goldprice-view .goldprice-td")][2:6]

    return Price24K(
        buy_bullion, sell_bullion, buy_ring, sell_ring, taken_on=datetime.datetime.now(), vendor="DOJI"
    )

In [15]:
async def pnj_today(session: aiohttp.ClientSession) -> Price24K:
    buy_bullion: int = 0
    sell_bullion: int = 0
    buy_ring: int = 0
    sell_ring: int = 0
    data = await (await session.get("https://edge-api.pnj.io/ecom-frontend/v1/get-gold-price?zone=00")).json()
    for i in data["data"]:
        if i["masp"] == "N24K":
            sell_ring = i["giaban"]*10000
            buy_ring = i["giamua"]*10000
        if i["masp"] == "KB":
            sell_bullion = i["giaban"]*10000
            buy_bullion = i["giamua"]*10000
    return Price24K(
        buy_bullion, sell_bullion, buy_ring, sell_ring, taken_on=datetime.datetime.now(), vendor="PNJ"
    )

In [16]:
async def btmc_today(session: aiohttp.ClientSession) -> Price24K:
    soup = bs4.BeautifulSoup(await (await session.get("https://btmc.vn/gia-vang-theo-ngay.html")).text(), "html.parser")
    bullion = list(list(soup.find_all(class_="bd_price_home")[0].children)[1].children)[3]  # type: ignore
    ring = list(list(soup.find_all(class_="bd_price_home")[0].children)[1].children)[4]  # type: ignore
    return Price24K (
        taken_on=datetime.datetime.now(),
        buy_bullion = int(list(bullion.children)[7].getText().strip()) * 10000,
        sell_bullion = int(list(bullion.children)[9].getText().strip()) * 10000,
        buy_ring = int(list(ring.children)[5].getText().strip()) * 10000,
        sell_ring = int(list(ring.children)[7].getText().strip()) * 10000,
        vendor="BTMC"
    )

In [17]:
async def btmh_today(session: aiohttp.ClientSession) -> Price24K:
    soup = bs4.BeautifulSoup(await (await session.get("https://baotinmanhhai.vn/gia-vang-hom-nay", ssl=False)).text(), "html.parser")
    buy_bullion: int = 0
    sell_bullion: int = 0
    buy_ring: int = 0
    sell_ring: int = 0

    for i in list(filter(lambda x: True if x.select(".items-center") else False,soup.select("div.flex div.grid")))[:-3]:
        if i.contents[0].contents[1].text == "Nhẫn Tròn ép vỉ (Kim Gia Bảo ) 24K (999.9)":
            sell_ring = int(i.contents[1].contents[0].contents[0].text.replace(".", "")) * 10
            buy_ring = int(i.contents[2].contents[0].contents[0].text.replace(".", "")) * 10

        if i.contents[0].contents[1].text == "Đồng vàng Kim Gia Bảo hoa sen":
            sell_bullion = int(i.contents[1].contents[0].contents[0].text.replace(".", "")) * 10
            buy_bullion = int(i.contents[2].contents[0].contents[0].text.replace(".", "")) * 10

    return Price24K(    
        taken_on=datetime.datetime.now(),
        buy_bullion=buy_bullion,
        sell_bullion=sell_bullion,
        buy_ring=buy_ring,
        sell_ring=sell_ring,
        vendor="BTMH"
    )

In [18]:
async def pqg_today(session: aiohttp.ClientSession) -> Price24K:
    soup = bs4.BeautifulSoup(await (await session.get("https://phuquygroup.vn/")).text(), "html.parser")
    price_table = list(soup.select_one("#priceList table").select("tr"))  # type: ignore
    return Price24K (
        taken_on=datetime.datetime.now(),
        buy_ring = int(price_table[2].select_one("td.buy-price").text.strip().replace(",", "")) * 10,  # type: ignore
        sell_ring = int(price_table[2].select_one("td.sell-price").text.strip().replace(",", "")) * 10,  # type: ignore
        buy_bullion = int(price_table[3].select_one("td.buy-price").text.strip().replace(",", "")) * 10,  # type: ignore
        sell_bullion = int(price_table[3].select_one("td.sell-price").text.strip().replace(",", "")) * 10,  # type: ignore
        vendor="PQG",
    )

In [19]:
async def mihong_today(session: aiohttp.ClientSession) -> Price24K:
    buy_bullion = 0
    sell_bullion = 0
    buy_ring = 0
    sell_ring = 0
    data = await (await session.get("https://api.mihong.vn/v1/gold-prices?market=domestic")).json()
    for i in data:
        if i["code"] == "999":
            buy_bullion = buy_ring = i["buyingPrice"] * 10
            sell_ring = sell_bullion = i["sellingPrice"] * 10


    return Price24K(
        taken_on=datetime.datetime.now(),
        buy_bullion=buy_bullion,
        sell_bullion=sell_bullion,
        buy_ring=buy_ring,
        sell_ring=sell_ring,
        vendor="MHG"
    )

In [20]:
async def ngoctham_today(session: aiohttp.ClientSession) -> Price24K:
    s = bs4.BeautifulSoup(await (await session.get("https://ngoctham.com/bang-gia-vang/")).text())
    buy_bullion = 0
    sell_bullion = 0
    buy_ring = 0
    sell_ring = 0

    for i in list(s.select("table.table tr"))[3:]:
        if list(i.find_all("td", attrs={"class": "name-gold"}))[0].text == "Vàng Ta 999.9":
            buy_bullion = int(list(i.find_all("td", attrs={"class": "price-buy"}))[0].text.replace(".", "")) * 10
            sell_bullion = int(list(i.find_all("td", attrs={"class": "price-sell"}))[0].text.replace(".", "")) * 10
        if list(i.find_all("td", attrs={"class": "name-gold"}))[0].text == "Nhẫn 999.9":
            buy_ring = int(list(i.find_all("td", attrs={"class": "price-buy"}))[0].text.replace(".", "")) * 10
            sell_ring = int(list(i.find_all("td", attrs={"class": "price-sell"}))[0].text.replace(".", "")) * 10

    return Price24K(
        taken_on=datetime.datetime.now(),
        buy_bullion=buy_bullion,
        sell_bullion=sell_bullion,
        buy_ring=buy_ring,
        sell_ring=sell_ring,
        vendor="NTH"
    )

In [ ]:
async def do_fetch(session: aiohttp.ClientSession, fetch_method: typing.Callable[[aiohttp.ClientSession], typing.Awaitable[Price24K]]) -> tuple[Price24K, str] | None:
    results = []
    async def fetch(tries = 10) -> Price24K | None:
        try:
            return await fetch_method(session)
        except:
            if tries > 0:
                print(f"[!] fetching failed for {fetch_method.__name__}. Retrying after 5s (attempt {10 - tries + 1}/10).")
                traceback.print_exc()
                time.sleep(5)
                return await fetch(tries-1)
            
    d = await fetch()
    if d:
        print(f"[*] fetched    {d.vendor:>4}  {d.buy_bullion}      {d.sell_bullion}       {d.buy_ring}     {d.sell_ring}")
        return d, f"{d.taken_on.isoformat()},{d.vendor},{d.buy_bullion},{d.sell_bullion},{d.buy_ring},{d.sell_ring}\n"
    return None
async def fetch_all(session: aiohttp.ClientSession) -> list[Price24K]:
    results = []
    print(f"            vendor  buy bullion    sell bullion    buy ring      sell ring")
    #       [*] fetched    SJC  173000000      173000000       173000000     173000000
    # with ThreadPoolExecutor(max_workers=8) as executor:
    #     r = [i for i in executor.map(fetch_thread, (sjc_today, doji_today, pnj_today, btmc_today, btmh_today, pqg_today, mihong_today, ngoctham_today)) if i is not None]
    r = await asyncio.gather(*[
        do_fetch(session, i) for i in (sjc_today, doji_today, pnj_today, btmc_today, btmh_today, pqg_today, mihong_today, ngoctham_today)
    ])
    results, csvdata = zip(*r)
    results, csvdata = list(results), "".join(csvdata)
    async with aiofiles.open(os.path.join("data", "snapshotdb.csv"), "a", encoding="utf8") as f:
        await f.write(csvdata)
    return results

async def run_cron():
    async with aiohttp.ClientSession() as session:
        async with aiofiles.open(os.path.join("data", "master.jsonl"), "a", encoding="utf8") as f:
            f.buffer.write(orjson.dumps({
                "takenOn": datetime.datetime.now().isoformat(),
                "prices": await fetch_all(session)
            }))
            f.buffer.write(b"\n")
            await f.flush()

In [22]:
await run_cron()

            vendor  buy bullion    sell bullion    buy ring      sell ring
[*] fetched     PQG  153500000      156500000       153500000     156500000
[*] fetched     MHG  153800000      155500000       153800000     155500000
[*] fetched     SJC  153500000      156500000       153500000     156500000
[*] fetched     PNJ  153500000      156500000       153500000     156500000
[*] fetched    BTMC  153500000      156500000       153500000     156500000
[*] fetched     NTH  143000000      146500000       143000000     146500000
[*] fetched    DOJI  153500000      156500000       153500000     156500000
[*] fetched    BTMH  153500000      156500000       153500000     156500000
